# 同音校正放寬（聲調／台灣混音）：開發集與誤改檢查

候選規則：同音教會專名校正另外接受「聲調不同」或台灣華語常見混音（sh/s、zh/z、ch/c、n/l、f/h、r/l、eng/en、ing/in、ang/an），
但改動的字不能在一般詞語裡（含頭尾和鄰字成詞、保護詞），原字和專名完全相同的字要至少和改動的字一樣多，另一個同長度專名也一樣符合時不改。例：招會→召會、經論→經綸、聲言者→申言者、伯薩撒→伯沙撒；
不改：祭司的、包羅萬有、生養眾多。網站目前**沒有**使用這條規則。

**不需要 GPU，也不用 Cloudflare 額度**：沿用雲端硬碟上已有的辨識結果，只重新計分。

1. `STAGE='dev'`：生命讀經開發集 16 篇（上次的 Whisper 停頓切段結果）＋恢復本／生命讀經正確全文的誤改檢查 → 把結果貼回對話（或告訴我「直接讀」）。
2. 開發集通過、我們看過誤改清單後，`STAGE='meet'`、`CONFIRM_MEET=True` 只跑一次驗收。**MEET 是封存測試，不能依它的結果再改規則。**

In [ ]:
# 第 1 格：設定
STAGE = 'dev'          # 'dev'（開發集＋誤改檢查）或 'meet'（封存測試驗收，最後才用）
CONFIRM_MEET = False   # 驗收 MEET 時改成 True

In [ ]:
# 第 2 格：連接 Google Drive、找到資料
import os, subprocess, sys
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive')
ARCHIVE = ROOT/'twgbr-archive'
if not (ARCHIVE/'life-study').is_dir():
    found = [Path(d) for d, dirs, _ in os.walk(ROOT) if len(Path(d).relative_to(ROOT).parts) < 3 and 'life-study' in dirs and 'recovery-version' in dirs]
    if not found:
        raise RuntimeError('找不到 twgbr-archive（含 life-study 與 recovery-version）')
    ARCHIVE = found[0]

def find(name):
    for path in (ROOT/name, ARCHIVE/name):
        if path.is_dir():
            return path
    found = [Path(d)/name for d, dirs, _ in os.walk(ROOT) if len(Path(d).relative_to(ROOT).parts) < 3 and name in dirs]
    if not found:
        raise RuntimeError(f'找不到 {name} 資料夾')
    return found[0]

def latest_run(folder, pattern):
    runs = sorted(p for p in (folder/'eval-runs').glob('*') if p.is_dir() and any(p.glob(pattern))) if (folder/'eval-runs').is_dir() else []
    if not runs:
        raise RuntimeError(f'{folder}/eval-runs 沒有 {pattern} 的辨識結果')
    return runs[-1]

if STAGE == 'dev':
    DATA = find('life-study-dev-v1')
    EARLIER = latest_run(DATA, '*.whisper_live_vad_10s.json')
    CONDITIONS, EXTRA = 'vad10', ['--development', '--loose-reference']
elif STAGE == 'meet':
    if not CONFIRM_MEET:
        raise RuntimeError('MEET 是封存測試，只在開發集通過後驗收一次；確定要跑請把 CONFIRM_MEET 改成 True')
    DATA = find('MEET')
    EARLIER = latest_run(DATA, '*.workers_ai_live_vad_10s.json')
    CONDITIONS, EXTRA = 'cfvad10,cfvad30,full', []
else:
    raise RuntimeError("STAGE 只能是 'dev' 或 'meet'")
print('資料：', DATA)
print('沿用辨識結果：', EARLIER)

In [ ]:
# 第 3 格：取得固定版本的程式與套件（約 1 分鐘）
COMMIT = 'b588d65f6e7bed9d44c6185602329712e8fee2aa'
REPO = Path('/content/Meeting-Live-Captions')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/timo893131/Meeting-Live-Captions.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', '-q', 'origin', COMMIT], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '-q', '--detach', COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rapidfuzz==3.14.1', 'opencc-python-reimplemented==0.1.7',
                'nodejs-wheel-binaries==22.20.0'], check=True)
import nodejs_wheel
NODE = str(Path(nodejs_wheel.__file__).parent/'bin'/'node')
os.environ['NODE_BINARY'] = NODE
print('程式版本', COMMIT[:12])

In [ ]:
# 第 4 格：重新計分（不重新辨識；約 2–5 分鐘）
RUN = DATA/'eval-runs'/(datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S') + '-near')
command = [sys.executable, '-u', str(REPO/'scripts/evaluate_meet.py'), '--meet', str(DATA), '--output', str(RUN),
           '--reuse', str(EARLIER), '--conditions', CONDITIONS, '--phonetic-near'] + EXTRA
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait():
    raise RuntimeError('計分失敗，請看上面的錯誤訊息')
print('\n結果：', RUN/'summary.md', '\n請看 phonetic（網站現在）對 phonetic_near（新規則）那兩行。')

In [ ]:
# 第 5 格（只有 dev）：誤改檢查——拿已經正確的恢復本＋生命讀經全文跑校正，新規則多改的地方都是誤改（約 10–20 分鐘）
import json
if STAGE == 'dev':
    folders = [str(ARCHIVE/'recovery-version/text'), str(ARCHIVE/'life-study/text')]
    def precision(*flags):
        out = subprocess.run([NODE, '--max-old-space-size=8192', str(REPO/'scripts/term-precision.mjs'), *folders, *flags],
                             capture_output=True, text=True, check=True, cwd=REPO).stdout
        return json.loads(out)
    before, after = precision(), precision('--near')
    rows = []
    for term, info in after.items():
        extra = info['changes'] - before.get(term, {}).get('changes', 0)
        if extra > 0:
            rows.append((extra, term, info['heard'], info.get('context', [])))
    rows.sort(reverse=True)
    total_before = sum(v['changes'] for v in before.values()); total_after = sum(v['changes'] for v in after.values())
    lines = [f'正確全文的改動：網站現在 {total_before} 處 → 新規則 {total_after} 處（多 {total_after - total_before} 處）', '',
             '| 多改次數 | 改成 | 原文寫法（例） | 前後文（例） |', '| --- | --- | --- | --- |']
    lines += [f"| {n} | {term} | {'、'.join(heard)} | {'、'.join(context)} |" for n, term, heard, context in rows[:80]]
    report = '\n'.join(lines)
    (RUN/'near-precision.md').write_text(report + '\n', encoding='utf-8')
    print(report)

## 請貼回對話（或回覆「直接讀」，我從雲端硬碟讀 summary.md 與 near-precision.md）

- 第 4 格最後的「專名比對（不需逐字對齊）」表（dev）或「合計」表（meet）：看 **phonetic** 與 **phonetic_near** 兩行。
- 第 5 格的誤改表：新規則在正確文字裡多改的地方。這些多半要擋掉（加入保護詞），或代表規則還太寬。